# Part 5

In [ ]:
import numpy as np
import pandas as pd
from final.lsm import lsm_price
from scipy.optimize import brentq
from joblib import Parallel, delayed
from final.garch import get_garch_parameters, get_garch_variances

## a) Implied Vol Calculation

``seed=42`` by default, so multiple sims can be used. Antithetic sampling is also used for all simulations to reduce variance. I also use ``joblib`` for parallel processing on ``df.apply`` since the MC and root finding takes a lot of time.

### i. Calculation

In [2]:
options = pd.read_parquet('../data/options.parquet')
options.head()

,symbol,date,expiry_date,days_to_expiry,strike_price,call_price,call_volume,put_price,put_volume,time_to_expiry,rf_rate,stock_price,stock_price_adj,dividend,dividend_yield,call_implied_vol,put_implied_vol
0,AAPL,2023-10-02,2024-03-15,165.0,160.0,21.80,0.0,5.38,1.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.183057,0.231396
1,AAPL,2023-10-02,2024-03-15,165.0,165.0,17.85,0.0,7.00,2.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.171455,0.229463
2,AAPL,2023-10-02,2024-03-15,165.0,170.0,15.95,9.0,9.05,55.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.196592,0.231396
3,AAPL,2023-10-02,2024-03-15,165.0,175.0,13.09,101.0,11.22,12.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.192725,0.229463
4,AAPL,2023-10-02,2024-03-15,165.0,180.0,10.20,51.0,13.34,1.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.186924,0.217861


Now we use our root-finding algorithm to find the call and put implied volatilities.

In [ ]:
def calculate_call_implied_vol(row):
    # define initial params
    S0 = row['stock_price']
    K = row['strike_price']
    T = row['time_to_expiry']
    r = row['rf_rate']
    q = row['dividend_yield']
    drift = r - q # risk neutral

    call_price_actual = row['call_price']

    # objective function (solve for sigma that gets market price)
    def objective(sigma):
        price_sim, _ = lsm_price(K=K, r=r, T=T, payoff='call', S0=S0, 
                             mu=drift, sigma=sigma, m=100000, n=100, seed=42, antithetic=True) # use antithetic to reduce variance
        return price_sim - call_price_actual 
    
    try:
        iv = brentq(objective, a=0.001, b=5.0, xtol=1e-3, maxiter=750) # use brent's method to solve for IV
        return iv
    except Exception as e:
        print(f"An error occurred: {e}")
        return np.nan

def calculate_put_implied_vol(row):
    S0 = row['stock_price']
    K = row['strike_price']
    T = row['time_to_expiry']
    r = row['rf_rate']
    q = row['dividend_yield']
    drift = r - q # risk neutral measure

    put_price_actual = row['put_price']

    def objective(sigma):
        price_sim, _ = lsm_price(K=K, r=r, T=T, payoff='put', S0=S0, 
                             mu=drift, sigma=sigma, m=100000, n=100, seed=42, antithetic=True) # use antithetic to reduce variance
        return price_sim - put_price_actual
    
    try:
        iv = brentq(objective, a=0.001, b=5.0, xtol=1e-3, maxiter=750)
        return iv
    except Exception:
        print(f"An error occurred: {e}")
        return np.nan

In [4]:
rows = options.to_dict('records') # for speed
# using parallel since it's a bit slow
call_results = Parallel(n_jobs=-1, verbose=5)(
    delayed(calculate_call_implied_vol)(row) for row in rows
)
options['call_implied_vol_calc'] = call_results

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   57.4s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:  2.9min
[Parallel(n_jobs=-1)]: Done 218 out of 218 | elapsed:  4.4min finished


In [5]:
put_results = Parallel(n_jobs=-1, verbose=5)(
    delayed(calculate_put_implied_vol)(row) for row in rows
)
options['put_implied_vol_calc'] = put_results

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:  3.1min
[Parallel(n_jobs=-1)]: Done 218 out of 218 | elapsed:  5.4min finished


In [6]:
options.head()

,symbol,date,expiry_date,days_to_expiry,strike_price,call_price,call_volume,put_price,put_volume,time_to_expiry,rf_rate,stock_price,stock_price_adj,dividend,dividend_yield,call_implied_vol,put_implied_vol,call_implied_vol_calc,put_implied_vol_calc
0,AAPL,2023-10-02,2024-03-15,165.0,160.0,21.80,0.0,5.38,1.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.183057,0.231396,0.181976,0.235150
1,AAPL,2023-10-02,2024-03-15,165.0,165.0,17.85,0.0,7.00,2.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.171455,0.229463,0.172320,0.233455
2,AAPL,2023-10-02,2024-03-15,165.0,170.0,15.95,9.0,9.05,55.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.196592,0.231396,0.198357,0.234681
3,AAPL,2023-10-02,2024-03-15,165.0,175.0,13.09,101.0,11.22,12.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.192725,0.229463,0.195648,0.231489
4,AAPL,2023-10-02,2024-03-15,165.0,180.0,10.20,51.0,13.34,1.0,0.654762,0.0555,173.75,171.916306,0.0,0.0058,0.186924,0.217861,0.186888,0.221243


In [7]:
options[options['call_implied_vol_calc'].isna()]

,symbol,date,expiry_date,days_to_expiry,strike_price,call_price,call_volume,put_price,put_volume,time_to_expiry,rf_rate,stock_price,stock_price_adj,dividend,dividend_yield,call_implied_vol,put_implied_vol,call_implied_vol_calc,put_implied_vol_calc
29,AAPL,2023-10-05,2024-03-15,162.0,155.0,24.60,0.0,4.3,46.0,0.642857,0.0551,174.91,173.064064,0.0,0.0058,NaN,0.250732,NaN,0.250432
49,AAPL,2023-10-09,2024-03-15,158.0,155.0,28.58,0.0,3.5,149.0,0.626984,0.0552,178.99,177.101005,0.0,0.0058,NaN,0.258467,NaN,0.251684


In [8]:
options[options['put_implied_vol_calc'].isna()]

,symbol,date,expiry_date,days_to_expiry,strike_price,call_price,call_volume,put_price,put_volume,time_to_expiry,rf_rate,stock_price,stock_price_adj,dividend,dividend_yield,call_implied_vol,put_implied_vol,call_implied_vol_calc,put_implied_vol_calc
137,AAPL,2023-10-19,2024-03-15,148.0,195.0,4.95,41.0,19.11,0.0,0.587302,0.0554,175.46,173.608260,0.0,0.0058,0.194658,NaN,0.189225,NaN
146,AAPL,2023-10-20,2024-03-15,147.0,190.0,5.70,67.0,15.64,0.0,0.583333,0.0552,172.88,171.055488,0.0,0.0058,0.200459,NaN,0.195365,NaN
147,AAPL,2023-10-20,2024-03-15,147.0,195.0,4.10,30.0,19.11,0.0,0.583333,0.0552,172.88,171.055488,0.0,0.0058,0.192725,NaN,0.188758,NaN
148,AAPL,2023-10-20,2024-03-15,147.0,200.0,2.85,64.0,27.00,1.0,0.583333,0.0552,172.88,171.055488,0.0,0.0058,0.184990,0.13665,0.182910,NaN
157,AAPL,2023-10-23,2024-03-15,144.0,195.0,3.75,84.0,19.11,0.0,0.571429,0.0554,173.00,171.174222,0.0,0.0058,0.183057,NaN,0.182248,NaN
167,AAPL,2023-10-24,2024-03-15,143.0,195.0,3.71,844.0,19.11,0.0,0.567460,0.0555,173.44,171.609578,0.0,0.0058,0.181123,NaN,0.179265,NaN
178,AAPL,2023-10-25,2024-03-15,142.0,200.0,2.24,58.0,28.55,1.0,0.563492,0.0555,171.10,169.294273,0.0,0.0058,0.179189,NaN,0.179608,NaN
186,AAPL,2023-10-26,2024-03-15,141.0,190.0,3.44,235.0,20.55,0.0,0.559524,0.0553,166.89,165.128704,0.0,0.0058,0.190791,NaN,0.190417,NaN
188,AAPL,2023-10-26,2024-03-15,141.0,200.0,1.67,147.0,28.55,0.0,0.559524,0.0553,166.89,165.128704,0.0,0.0058,0.183057,NaN,0.184469,NaN


### ii. Questions

* Yes, there are records where implied volatility could not be computed. For most of these cases, the options are deep in-the-money American options whose prices are dominated by intrinsic value and early exercise, making the option value nearly insensitive to volatility and therefore not solvable for implied volatility. Additionally, trading volume is coincidentally very low for the NaNs, so even though it doesn't enter the calculation, it may indicate stale data. As a check, note that all but one of the options where I was not able to calculate IV also had NaN IVs in the actual implied volatility column. 
* The implied volatilities are **not constant**. The Black-Scholes-Merton framework assumes that implied volatility is a constant parameter, but as we can see in the data, this is not the case. However, our data violates the assumptions significantly, as BSM assumes no early-exercise (our options are American), market friction does exist, and the underlying might not be lognormally distributed (in returns). Thus, it is very plausible that the IVs will not follow the BSM framework and be a constant value.

## b) Greeks

### Delta

We can estimate $\Delta$ by using the central difference approximation $$\Delta \approx \frac{V(S+dS) - V(S-dS)}{2dS}.$$

In [ ]:
def calc_call_delta(row):
    # define initial params
    S0 = row['stock_price']
    dS = 0.01 * S0 # define the "bump"
    K = row['strike_price']
    T = row['time_to_expiry']
    r = row['rf_rate']
    q = row['dividend_yield']
    drift = r - q
    sigma = row['call_implied_vol_calc']

    price_sim_up, _ = lsm_price(K=K, r=r, T=T, payoff='call', S0=S0+dS, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    price_sim_dn, _ = lsm_price(K=K, r=r, T=T, payoff='call', S0=S0-dS, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    return (price_sim_up - price_sim_dn) / (2 * dS) # approximation using central difference

def calc_put_delta(row):
    S0 = row['stock_price']
    dS = 0.01 * S0
    K = row['strike_price']
    T = row['time_to_expiry']
    r = row['rf_rate']
    q = row['dividend_yield']
    drift = r - q
    sigma = row['put_implied_vol_calc']

    price_sim_up, _ = lsm_price(K=K, r=r, T=T, payoff='put', S0=S0+dS, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    price_sim_dn, _ = lsm_price(K=K, r=r, T=T, payoff='put', S0=S0-dS, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    return (price_sim_up - price_sim_dn) / (2 * dS)

In [10]:
rows = options.to_dict('records')
call_delta = Parallel(n_jobs=-1, verbose=5)(
    delayed(calc_call_delta)(row) for row in rows
)
options['call_delta'] = call_delta

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   16.4s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:   44.7s
[Parallel(n_jobs=-1)]: Done 218 out of 218 | elapsed:  1.1min finished


In [11]:
put_delta = Parallel(n_jobs=-1, verbose=5)(
    delayed(calc_put_delta)(row) for row in rows
)
options['put_delta'] = put_delta

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   15.7s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:   42.3s
[Parallel(n_jobs=-1)]: Done 218 out of 218 | elapsed:  1.2min finished


### Gamma

We can approximate $\Gamma$ using the second-order approximation $$\Gamma \approx \frac{V(S+dS) + V(S-dS) - 2V(S)}{dS^2}.$$

In [ ]:
def calc_call_gamma(row):
    # define initial params
    S0 = row['stock_price']
    dS = 0.01 * S0
    K = row['strike_price']
    T = row['time_to_expiry']
    r = row['rf_rate']
    q = row['dividend_yield']
    drift = r - q
    sigma = row['call_implied_vol_calc']

    price_sim, _ = lsm_price(K=K, r=r, T=T, payoff='call', S0=S0, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    price_sim_up, _ = lsm_price(K=K, r=r, T=T, payoff='call', S0=S0+dS, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    price_sim_dn, _ = lsm_price(K=K, r=r, T=T, payoff='call', S0=S0-dS, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    return (price_sim_up + price_sim_dn - 2 * price_sim) / (dS**2) # approximation with finite diffs

def calc_put_gamma(row):
    # define initial params
    S0 = row['stock_price']
    dS = 0.01 * S0
    K = row['strike_price']
    T = row['time_to_expiry']
    r = row['rf_rate']
    q = row['dividend_yield']
    drift = r - q
    sigma = row['put_implied_vol_calc']

    price_sim, _ = lsm_price(K=K, r=r, T=T, payoff='put', S0=S0, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    price_sim_up, _ = lsm_price(K=K, r=r, T=T, payoff='put', S0=S0+dS, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    price_sim_dn, _ = lsm_price(K=K, r=r, T=T, payoff='put', S0=S0-dS, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    return (price_sim_up + price_sim_dn - 2 * price_sim) / (dS**2)

In [13]:
rows = options.to_dict('records')
call_gamma = Parallel(n_jobs=-1, verbose=5)(
    delayed(calc_call_gamma)(row) for row in rows
)
options['call_gamma'] = call_gamma

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   22.3s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 218 out of 218 | elapsed:  1.6min finished


In [14]:
put_gamma = Parallel(n_jobs=-1, verbose=5)(
    delayed(calc_put_gamma)(row) for row in rows
)
options['put_gamma'] = put_gamma

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   23.3s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:  1.1min
[Parallel(n_jobs=-1)]: Done 218 out of 218 | elapsed:  1.7min finished


### Theta

We can estimate $\theta$ by simulating time and then using the finite difference approximation $$\theta \approx \frac{V(T-(t+dt)) - V(T-t)}{dt}.$$

In [ ]:
def calc_call_theta(row):
    # define initial params
    S0 = row['stock_price']
    dt = 1/252
    K = row['strike_price']
    T = row['time_to_expiry']
    r = row['rf_rate']
    q = row['dividend_yield']
    drift = r - q
    sigma = row['call_implied_vol_calc']

    price_sim_forward, _ = lsm_price(K=K, r=r, T=T-dt, payoff='call', S0=S0, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    price_sim, _ = lsm_price(K=K, r=r, T=T, payoff='call', S0=S0, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    return (price_sim_forward - price_sim) / dt # forward diff approximation

def calc_put_theta(row):
    # define initial params
    S0 = row['stock_price']
    dt = 1/252
    K = row['strike_price']
    T = row['time_to_expiry']
    r = row['rf_rate']
    q = row['dividend_yield']
    drift = r - q
    sigma = row['put_implied_vol_calc']

    price_sim_forward, _ = lsm_price(K=K, r=r, T=T-dt, payoff='put', S0=S0, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    price_sim, _ = lsm_price(K=K, r=r, T=T, payoff='put', S0=S0, mu=drift, sigma=sigma, m=100000, n=100, antithetic=True)
    return (price_sim_forward - price_sim) / dt

In [16]:
rows = options.to_dict('records')
call_theta = Parallel(n_jobs=-1, verbose=5)(
    delayed(calc_call_theta)(row) for row in rows
)
options['call_theta'] = call_theta

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   15.0s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:   42.7s
[Parallel(n_jobs=-1)]: Done 218 out of 218 | elapsed:  1.1min finished


In [18]:
put_theta = Parallel(n_jobs=-1, verbose=5)(
    delayed(calc_put_theta)(row) for row in rows
)
options['put_theta'] = put_theta

[Parallel(n_jobs=-1)]: Using backend LokyBackend with 16 concurrent workers.
[Parallel(n_jobs=-1)]: Done  40 tasks      | elapsed:   21.7s
[Parallel(n_jobs=-1)]: Done 130 tasks      | elapsed:   52.9s
[Parallel(n_jobs=-1)]: Done 218 out of 218 | elapsed:  1.4min finished


In [ ]:
# converting theta to day
options['call_theta_day'] = options['call_theta'] / 252
options['put_theta_day'] = options['put_theta'] / 252

In [20]:
options.head()

,symbol,date,expiry_date,days_to_expiry,strike_price,call_price,call_volume,put_price,put_volume,time_to_expiry,...,call_implied_vol_calc,put_implied_vol_calc,call_delta,put_delta,call_gamma,put_gamma,call_theta,put_theta,call_theta_day,put_theta_day
0,AAPL,2023-10-02,2024-03-15,165.0,160.0,21.80,0.0,5.38,1.0,0.654762,...,0.181976,0.235150,0.801081,-0.251845,0.010637,0.010738,-11.102523,-6.236961,-0.044058,-0.024750
1,AAPL,2023-10-02,2024-03-15,165.0,165.0,17.85,0.0,7.00,2.0,0.654762,...,0.172320,0.233455,0.747917,-0.306707,0.013317,0.009976,-11.280816,-6.151993,-0.044765,-0.024413
2,AAPL,2023-10-02,2024-03-15,165.0,170.0,15.95,9.0,9.05,55.0,0.654762,...,0.198357,0.234681,0.656309,-0.367022,0.010929,0.017556,-12.947679,-7.124547,-0.051380,-0.028272
3,AAPL,2023-10-02,2024-03-15,165.0,175.0,13.09,101.0,11.22,12.0,0.654762,...,0.195648,0.231489,0.589374,-0.438493,0.010289,0.018505,-12.322506,-6.748043,-0.048899,-0.026778
4,AAPL,2023-10-02,2024-03-15,165.0,180.0,10.20,51.0,13.34,1.0,0.654762,...,0.186888,0.221243,0.519824,-0.509437,0.018208,0.016534,-11.777082,-5.611141,-0.046734,-0.022266


## c) GARCH Realized Vol Estimates

We need the return and variance from the last date, so we load in the stock price series.

In [ ]:
stock = pd.read_parquet('../data/stock.parquet')
stock['log_ret'] = np.log(stock['adj_close']).diff()
# Fitting garch and getting variances over fitting period
params = get_garch_parameters(stock['log_ret'][1:])
sigma_sq = get_garch_variances(stock['log_ret'][1:], params)

c:\Users\rahul\anaconda3\Lib\site-packages\scipy\optimize\_slsqp_py.py:435: RuntimeWarning: Values in x were outside bounds during a minimize step, clipping to bounds
  fx = wrapped_fun(x)


In [ ]:
last_var, last_ret = sigma_sq[-1], stock['log_ret'].iloc[-1] # 2023-10-02, same date as our options start
dates = np.unique(options['date'])
last_price = stock['adj_close'].iloc[-1]
prices_series = [options[options['date'] == d]['stock_price_adj'].iloc[0] for d in dates] # making full price series, just use first upx from each date (it won't change)
prices_series = pd.Series(prices_series, index=dates)
rets = (np.log(prices_series).diff()) * 100 # what I used while fitting
rets.fillna(last_ret, inplace=True)

In [ ]:
a0 = params[0]
a1 = params[1]
b1 = params[2]

forecasts = {}

for date in dates:
    # forecast
    current_var = a0 + a1 * last_ret**2 + b1 * last_var
    annualized = np.sqrt(current_var * 252) / 100 # convert to annualized
    forecasts[date] = annualized
    last_ret = rets.loc[date]
    last_var = current_var
forecasts = pd.Series(forecasts)
vol_df = forecasts.reset_index()
vol_df.columns = ['date', 'realized_vol']
options = options.merge(vol_df, on='date', how='left') # use main df as master

In [24]:
options.head()

,symbol,date,expiry_date,days_to_expiry,strike_price,call_price,call_volume,put_price,put_volume,time_to_expiry,...,put_implied_vol_calc,call_delta,put_delta,call_gamma,put_gamma,call_theta,put_theta,call_theta_day,put_theta_day,realized_vol
0,AAPL,2023-10-02,2024-03-15,165.0,160.0,21.80,0.0,5.38,1.0,0.654762,...,0.235150,0.801081,-0.251845,0.010637,0.010738,-11.102523,-6.236961,-0.044058,-0.024750,0.221099
1,AAPL,2023-10-02,2024-03-15,165.0,165.0,17.85,0.0,7.00,2.0,0.654762,...,0.233455,0.747917,-0.306707,0.013317,0.009976,-11.280816,-6.151993,-0.044765,-0.024413,0.221099
2,AAPL,2023-10-02,2024-03-15,165.0,170.0,15.95,9.0,9.05,55.0,0.654762,...,0.234681,0.656309,-0.367022,0.010929,0.017556,-12.947679,-7.124547,-0.051380,-0.028272,0.221099
3,AAPL,2023-10-02,2024-03-15,165.0,175.0,13.09,101.0,11.22,12.0,0.654762,...,0.231489,0.589374,-0.438493,0.010289,0.018505,-12.322506,-6.748043,-0.048899,-0.026778,0.221099
4,AAPL,2023-10-02,2024-03-15,165.0,180.0,10.20,51.0,13.34,1.0,0.654762,...,0.221243,0.519824,-0.509437,0.018208,0.016534,-11.777082,-5.611141,-0.046734,-0.022266,0.221099


In [25]:
options.to_parquet('../data/options_final.parquet')

## d) Questions

In [26]:
options_final_iv = options.dropna()
put_higher = (options_final_iv['put_implied_vol_calc'] > options_final_iv['realized_vol']).mean()
call_higher = (options_final_iv['call_implied_vol_calc'] > options_final_iv['realized_vol']).mean()
print("Proportion of put IVs higher than GARCH vol: ", put_higher)
print("Proportion of call IVs higher than GARCH vol: ", call_higher)

Proportion of put IVs higher than GARCH vol:  0.8502415458937198
Proportion of call IVs higher than GARCH vol:  0.24154589371980675


To answer the first question, we can see that a majority of the calculated put implied volatilities are greater than the forecasted volatility using a GARCH(1, 1). However, the GARCH(1, 1) volatilities are usually higher than the call implied volatilities. This indicates that the market is "overpricing" puts relative to the price moves our GARCH model suggests, and investors are paying "extra" for the downside risk protection (Volatility Risk Premium). On the other hand, the market is "underpricing" calls.

Yes, there is potential to make a profit through delta-hedging. In the Black-Scholes world, the total PnL from delta hedging can be predicted by the identity $P=\nu \cdot (\sigma_r - \sigma_i)$. While our situation is different (real-world American options), a similar concept applies. If we decided to take a position in the put options, since implied is generally higher than realized, we would short the option (negative $\nu$) and long the stock. We collect the higher premium (since IV is higher), and since the stock moves less than the market predicted it would, theta works in our favor and outweighs the costs of hedging. Conversely, if we decided to take a position in the call options, we would long the calls and short the underlying (since realized is generally greater than implied). We buy the call at a "cheaper" price and benefit from the higher-than-expected volatility, making it more likely the stock will go in-the-money.

The best options for a volatility-arbitrage strategy would be At-the-Money put options. There are significant discrepancies in volatility pricing between implied and realized when it comes to put options (85%). To understand where this comes into play, note that the instantaneous PnL of our portfolio, $dP$, from selling an option $V$ and delta-hedging it can be represented as $$dP = \frac{1}{2} \Gamma S^2 (\sigma_i^2 - \sigma_r^2).$$
There are two components in this equation that demonstrate why ATM put options would be the best candidates:
* $\Gamma$. Gamma is significantly higher for ATM options, as there is the most uncertainty, so small price moves drastically change $\Delta$. A higher $\Gamma$ would increase our instantaneous PnL ($dP$).
* $\sigma_i^2 - \sigma_r^2$. For over 85% of the options analyzed, the put implied volatility was higher than the realized volatility predicted by GARCH(1, 1). This makes the term positive and contributes to a greater $dP$.     

The final DataFrame with all calculations appended is stored in ``options_final.parquet``.